# litterbug — Colab walkthrough

Full pipeline in order: GPU check, install, data, validate, EDA, train both models, evaluate, error
analysis, inference.

Needs a GPU runtime (`Runtime -> Change runtime type -> T4 GPU`) and about three hours. `dataset/`,
`runs/` and `*.pt` are git-ignored, so the data and the checkpoints are both produced here.

## 1. Confirm the GPU

Ultralytics falls back to CPU silently, which turns three hours into several days.

In [ ]:
import torch

print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("vram:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    raise SystemExit("No GPU — set Runtime -> Change runtime type -> T4 GPU and re-run.")

## 2. Clone and install

The install is **editable** on purpose. `PROJECT_ROOT` in `src/litterbug/common/constants.py` is
`Path(__file__).resolve().parents[3]`, so only an editable install keeps `dataset/` and `runs/`
resolving inside the clone.

In [ ]:
REPO_URL = "https://github.com/strawberyy-coconut/litterbug.git"
WORKDIR = "/content/litterbug"

%cd /content
!test -d {WORKDIR} || git clone --depth 1 {REPO_URL}
%cd {WORKDIR}
%pip install -q -e .

In [ ]:
# Paths must resolve into the clone, not into site-packages.
from litterbug.common.constants import CLASS_NAMES, DATA_YAML, PROJECT_ROOT, RUNS_DIR

print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATA_YAML    :", DATA_YAML)
print("RUNS_DIR     :", RUNS_DIR)
print("classes      :", CLASS_NAMES)

assert "site-packages" not in str(PROJECT_ROOT), "not an editable install - re-run the cell above"

In [ ]:
# Resolved configuration and environment, no GPU time.
!litterbug train --task segment --dry-run

## 3. Fetch the dataset

BUU Waste Occlusion Dataset, VisionLab, Burapha University — **CC BY 4.0** (citation:
`report/report.md` §11). Not redistributed here: it comes from Kaggle and needs an API token,
`kaggle.json`.

In [ ]:
import shutil
from pathlib import Path

DATA_DIR = Path(WORKDIR) / "dataset" / "1_Model_Training_Data"
ARCHIVE = Path("/content/kaggle")

if not (DATA_DIR / "data.yaml").is_file():
    token = Path.home() / ".kaggle" / "kaggle.json"
    if not token.is_file():
        from google.colab import files

        print("Upload kaggle.json (Kaggle -> Account -> Create New API Token)")
        uploaded = files.upload()
        token.parent.mkdir(parents=True, exist_ok=True)
        token.write_bytes(next(iter(uploaded.values())))
        token.chmod(0o600)

    shutil.rmtree(ARCHIVE, ignore_errors=True)
    ARCHIVE.mkdir(parents=True, exist_ok=True)
    %pip install -q kaggle
    !kaggle datasets download -d visionlab1buu/buu-waste-occlusion-dataset -p {ARCHIVE} --unzip

    # The archive keeps the provider's layout; link data.yaml into the path the pipeline expects.
    found = [p.parent for p in ARCHIVE.rglob("data.yaml")]
    if not found:
        raise FileNotFoundError(f"No data.yaml under {ARCHIVE}; arrange it as {DATA_DIR}.")
    DATA_DIR.parent.mkdir(parents=True, exist_ok=True)
    if not DATA_DIR.exists():
        DATA_DIR.symlink_to(found[0])

print("dataset:", DATA_DIR.resolve())

## 4. Validate the data

Image/label pairing, polygon validity, coordinate ranges, class balance. Must reproduce 1,400 / 400 /
200 images and 17,400 / 4,941 / 2,507 instances.

In [ ]:
!litterbug validate

## 5. Exploratory analysis

Class counts, imbalance, geometry and lighting. Writes `runs/dataset-eda/dataset_eda.json`.

In [ ]:
!litterbug eda

## 6. Fine-tune both models

Same split, schedule and seed, so the head is the only difference. Hyperparameters are frozen in
`src/litterbug/common/config.py`; a one-epoch run is a plumbing check, not a preview.

In [ ]:
!litterbug train --task segment --name litterbug-segment-yolo26s-seg
!litterbug train --task detect --name litterbug-detect-yolo26s

## 7. Evaluate

`valid` is for iteration. `test` is touched once, at the end.

In [ ]:
from pathlib import Path


def newest_checkpoint(task: str) -> Path:
    """Most recent best.pt for a task, so the notebook does not hardcode a run name."""
    candidates = sorted(
        Path(RUNS_DIR).glob(f"*{task}*/weights/best.pt"), key=lambda p: p.stat().st_mtime
    )
    if not candidates:
        raise FileNotFoundError(f"no checkpoint under {RUNS_DIR} for {task!r}")
    return candidates[-1]


SEGMENT = newest_checkpoint("seg")
DETECT = newest_checkpoint("detect")
print("segment:", SEGMENT)
print("detect :", DETECT)

In [ ]:
!litterbug val --weights {SEGMENT} --split val

In [ ]:
# The held-out split. Run once.
!litterbug val --weights {SEGMENT} --split test
!litterbug val --weights {DETECT} --split test

## 8. Error analysis

Re-derives the per-instance matching Ultralytics does not expose: greedy, one-to-one, class-agnostic,
at mask IoU 0.5. `--iou-mode box` matches on boxes instead.

In [ ]:
!litterbug analyze --weights {SEGMENT} --split test
!litterbug analyze --weights {SEGMENT} --split test --iou-mode box

In [ ]:
from IPython.display import Image, Markdown, display

from litterbug.training.error_examples import ExampleConfig, examples

gallery = examples(ExampleConfig(weights=SEGMENT, split="test"))
display(Markdown(Path(gallery["commentary"]).read_text(encoding="utf-8")[:2000]))

first = sorted(Path(gallery["figures_dir"]).glob("*.png"))[0]
display(Image(filename=str(first)))

## 9. Inference

In [ ]:
from litterbug.running.predict import PredictConfig, predict

# First image, by name, from the held-out split.
sample = min(p for p in (DATA_DIR / "test" / "images").iterdir() if p.is_file())
result = predict(PredictConfig(weights=SEGMENT, source=sample, name="notebook-sample"))

print("kind       :", result.kind)
print("detections :", len(result.detections))
print("counts     :", result.counts)
print("artifacts  :")
for path in result.artifacts:
    print("  ", path)

Video and tracking (`--track`, ByteTrack) are CLI-only: the report's clips are licensed previews that
cannot be redistributed. Results in `report/report.md` §8.

## 10. Reference results

**Segmentation, `test` — 200 images / 2,507 instances**

| Metric | Box | Mask |
| --- | --- | --- |
| mAP50-95 | 0.837 | 0.796 |
| mAP50 | 0.954 | 0.955 |
| Precision | 0.939 | 0.944 |
| Recall | 0.901 | 0.905 |

Mean IoU over matched pairs: 0.901 mask, 0.924 box.

**Detection baseline, `test`:** box mAP50-95 0.8376, mAP50 0.9557, precision 0.944, recall 0.913.

Box-head difference between the two models: 0.0004. `valid` runs one to two points higher (box 0.857 /
mask 0.814), nearly all of the gap in the smallest size quartile.